# Inspect UMAP clusters of complex benchmark compounds

Uses compounds from the repository's frozen BACE-1, EGFR, Tox21, and ClinTox
snapshot. Compares HDBSCAN groups with Bemis-Murcko scaffolds.


In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit.Chem import Draw
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances, silhouette_score
from umap import UMAP

# Support launching Jupyter from the repository root, ai/, or ai/notebooks/.
ai_dir = next(
    (
        candidate
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "ai")
        if (candidate / "src" / "pipeline" / "split.py").is_file()
    ),
    None,
)
if ai_dir is None:
    raise RuntimeError("Open this notebook from within the CHM4930 repository.")
sys.path.insert(0, str(ai_dir / "src"))

from featurize.fingerprint import mol_to_ecfp4
from featurize.smiles import mol_to_smiles
from pipeline.preprocess import to_mol
from pipeline.split import umap_split

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
display(pd.Series({package: version(package) for package in ["umap-learn", "scikit-learn", "rdkit", "numpy"]}, name="version"))


## Settings

`CLUSTER_DIMS = 2` makes the scatter plot show the exact coordinates used by HDBSCAN.
The pipeline currently defaults to **five** dimensions. Set `CLUSTER_DIMS = 5` to inspect
that configuration; a PCA projection will then display the clustered embedding in 2D.
That projection can hide separations that exist in the higher-dimensional embedding.The other 
defaults below match `umap_split()`.

In [ ]:
SEED = 42
N_NEIGHBORS = 30
CLUSTER_DIMS = 2
MIN_CLUSTER_SIZE = 10
MIN_SAMPLES = 10  # scikit-learn counts the point itself
ANNOTATE_IDS = False
EXAMPLES_PER_CLUSTER = 6
MAX_CLUSTER_GRIDS = 8

# Build a larger, chemically varied panel from the local benchmark snapshot.
SAMPLE_PER_SOURCE = 100  # up to 400 unique molecules across the four sources
MIN_HEAVY_ATOMS = 20
MAX_HEAVY_ATOMS = 70
MIN_RINGS = 2
MIN_HETERO_ATOMS = 2
SCAFFOLDS_TO_HIGHLIGHT = 12

# Requested train / validation / test fractions for both comparison maps.
SPLIT_FRAC = (0.8, 0.1, 0.1)
#SPLIT_FRAC = (0.4, 0.3, 0.3)


## Complex compounds from the frozen dataset

Read only SMILES and source-membership flags from `ai/data/baseline/combined.csv`.
Standardize structures with the pipeline's `to_mol()`, exclude wildcard atoms, then
keep molecules with 20-70 heavy atoms, at least two rings, and at least two heteroatoms
(other than carbon/hydrogen). These are size/structure filters, not a drug-likeness score.

Sample up to `SAMPLE_PER_SOURCE` additional unique molecules per source with a fixed
seed. Overlapping source memberships are retained. The cells print the eligible and
selected counts so shortages are visible; filters are never silently relaxed. Molecule
names identify their original zero-based CSV data row, rather than inventing drug names.

In [ ]:
snapshot_path = ai_dir / "data" / "baseline" / "combined.csv"
source_names = {
    "in_bace1": "BACE-1",
    "in_egfr": "EGFR",
    "in_tox21": "Tox21",
    "in_clintox": "ClinTox",
}
source_flags = list(source_names)
snapshot = pd.read_csv(snapshot_path, usecols=["smiles", *source_flags])
snapshot.insert(0, "snapshot_row", np.arange(len(snapshot)))
snapshot[source_flags] = snapshot[source_flags].fillna(False).astype(bool)

eligible_records = []
invalid_count = 0
for row in snapshot.itertuples(index=False):
    try:
        mol = to_mol(row.smiles)
    except (ValueError, TypeError):
        invalid_count += 1
        continue
    if any(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms()):
        continue
    heavy_atoms = mol.GetNumHeavyAtoms()
    rings_count = mol.GetRingInfo().NumRings()
    hetero_atoms = sum(atom.GetAtomicNum() not in (1, 6) for atom in mol.GetAtoms())
    if not (
        MIN_HEAVY_ATOMS <= heavy_atoms <= MAX_HEAVY_ATOMS
        and rings_count >= MIN_RINGS
        and hetero_atoms >= MIN_HETERO_ATOMS
    ):
        continue
    canonical = mol_to_smiles(mol)
    record = row._asdict()
    record.update(
        name=f"snapshot row {row.snapshot_row}",
        mol=mol,
        canonical_smiles=canonical,
        heavy_atoms=heavy_atoms,
        rings=rings_count,
        hetero_atoms=hetero_atoms,
        has_stereochemistry="@" in canonical,
    )
    eligible_records.append(record)

eligible = pd.DataFrame(eligible_records)
if eligible.empty:
    raise ValueError("No compounds meet the configured complexity filters. Adjust the settings above.")
# Merge source memberships if standardization makes two original rows equivalent.
eligible[source_flags] = eligible.groupby("canonical_smiles")[source_flags].transform("any")
eligible = eligible.drop_duplicates("canonical_smiles").reset_index(drop=True)

selected = []
selected_smiles = set()
for offset, (flag, source_name) in enumerate(source_names.items()):
    available = eligible.loc[
        eligible[flag] & ~eligible["canonical_smiles"].isin(selected_smiles)
    ]
    sample = available.sample(n=min(SAMPLE_PER_SOURCE, len(available)), random_state=SEED + offset)
    print(f"{source_name}: {len(available)} eligible unselected compounds; selected {len(sample)}")
    selected.append(sample)
    selected_smiles.update(sample["canonical_smiles"])

molecules = pd.concat(selected, ignore_index=True)
if len(molecules) < max(3, MIN_CLUSTER_SIZE, MIN_SAMPLES):
    raise ValueError("Too few selected compounds for clustering. Adjust the sample size or filters.")
molecules["source"] = [
    " + ".join(title for flag, title in source_names.items() if record[flag])
    for record in molecules.to_dict("records")
]
molecules.insert(0, "id", np.arange(len(molecules)))
fingerprints = np.stack([mol_to_ecfp4(mol) for mol in molecules["mol"]])

print(f"\nSnapshot: {len(snapshot)} rows; eligible unique compounds: {len(eligible)}")
print(f"Selected panel: {len(molecules)} molecules; fingerprint matrix: {fingerprints.shape}")
print(f"{int(molecules['has_stereochemistry'].sum())} selected structures have atom stereochemistry")
if invalid_count:
    print(f"Skipped {invalid_count} unparseable structures before applying filters.")

display(pd.DataFrame({
    "eligible": eligible[source_flags].sum(),
    "selected memberships (may overlap)": molecules[source_flags].sum(),
}).rename(index=source_names))

display(molecules[["heavy_atoms", "rings", "hetero_atoms"]].describe().loc[["min", "50%", "max"]].T)
display(molecules.drop(columns="mol"))


## Fit real UMAP and HDBSCAN

These settings reproduce the operations inside `umap_split()`, with embedding dimensions
controlled above. Keeping the embedding here lets us inspect the coordinates directly.
Only the fingerprint matrix is supplied to either estimator.


In [ ]:
reducer = UMAP(
    n_neighbors=min(N_NEIGHBORS, len(molecules) - 1),
    min_dist=0.0,
    n_components=CLUSTER_DIMS,
    metric="jaccard",
    init="random",
    random_state=SEED,
    n_jobs=1,
)
embedding = reducer.fit_transform(fingerprints)
clusterer = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES)
molecules["group"] = clusterer.fit_predict(embedding)

if CLUSTER_DIMS == 2:
    plot_coordinates = embedding
    axis_labels = ("UMAP 1", "UMAP 2")
    view_description = "2D UMAP coordinates used for clustering"
elif CLUSTER_DIMS > 2:
    projection = PCA(n_components=2)
    plot_coordinates = projection.fit_transform(embedding)
    axis_labels = ("PC 1 of UMAP embedding", "PC 2 of UMAP embedding")
    view_description = f"2D PCA view of the {CLUSTER_DIMS}D UMAP embedding"
    print(f"PCA view explains {projection.explained_variance_ratio_.sum():.1%} of embedding variance")
else:
    raise ValueError("Set CLUSTER_DIMS to at least 2 for this notebook.")

molecules[["x", "y"]] = plot_coordinates
clustered = molecules["group"].ge(0).to_numpy()
cluster_ids = sorted(molecules.loc[clustered, "group"].unique())
print(f"{len(cluster_ids)} clusters; {int((~clustered).sum())} noise molecules; {clustered.mean():.1%} clustered")
if not cluster_ids:
    print("No clusters were found. Inspect the reference panel and try smaller HDBSCAN thresholds.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), layout="constrained")
cluster_palette = plt.get_cmap("tab20")
family_palette = plt.get_cmap("tab20")

for label in sorted(molecules["group"].unique()):
    subset = molecules[molecules["group"] == label]
    is_noise = label == -1
    axes[0].scatter(
        subset["x"], subset["y"],
        color="0.65" if is_noise else cluster_palette(int(label) % 20),
        marker="x" if is_noise else "o", s=48, alpha=0.85,
        label=("Noise" if is_noise else f"Cluster {label}") + f" ({len(subset)})",
    )
for color_id, (family, subset) in enumerate(molecules.groupby("source", sort=False)):
    axes[1].scatter(
        subset["x"], subset["y"], color=family_palette(color_id % 20),
        s=48, alpha=0.85, label=family,
    )
for ax in axes:
    ax.set(xlabel=axis_labels[0], ylabel=axis_labels[1])
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    if ANNOTATE_IDS:
        for row in molecules.itertuples():
            ax.annotate(str(row.id), (row.x, row.y), fontsize=6, xytext=(3, 3), textcoords="offset points")
axes[0].set_title("HDBSCAN clusters (gray = noise)")
axes[1].set_title("Dataset source membership; identical coordinates")
fig.suptitle(view_description, fontsize=14)
plt.show()


## Train / validation / test: scaffold versus UMAP splitting

Both maps show **the same molecules at the same coordinates**, colored by their actual
`train`, `val`, or `test` assignment, with the same `SPLIT_FRAC` and `SEED`.
Scaffold splitting retains its original per-source capacity rules.


In [ ]:
from textwrap import fill
from pipeline.split import SPLITS, murcko_scaffold, scaffold_split

# Retain scaffold metadata for the structure catalog below.
scaffold_view = molecules[["id", "name", "source", "smiles", "x", "y"]].copy()
scaffold_view["group"] = scaffold_view["smiles"].map(murcko_scaffold)
scaffold_counts = scaffold_view.loc[scaffold_view["group"] != "", "group"].value_counts()
scaffold_keys = sorted(scaffold_counts.index, key=lambda key: (-int(scaffold_counts[key]), key))
highlighted_scaffolds = scaffold_keys[:SCAFFOLDS_TO_HIGHLIGHT]
scaffold_names = {key: f"S{i + 1}" for i, key in enumerate(scaffold_keys)}
acyclic_scaffold_view = scaffold_view[scaffold_view["group"] == ""]

# Use the real public splitters; neither the embedding nor its coordinates change.
comparison_flags = [column for column in molecules if column.startswith("in_")]
split_input = molecules[["id", "name", "source", "smiles", *comparison_flags]].copy()
split_specs = [
    ("Scaffold", scaffold_split, {}),
    ("UMAP", umap_split, {
        "n_neighbors": N_NEIGHBORS,
        "n_components": CLUSTER_DIMS,
        "min_cluster_size": MIN_CLUSTER_SIZE,
        "min_samples": MIN_SAMPLES,
    }),
]
split_results = {}
split_errors = {}
for method, splitter, parameters in split_specs:
    try:
        split_results[method] = splitter(
            split_input, frac=SPLIT_FRAC, seed=SEED, **parameters
        )
    except ValueError as error:
        split_errors[method] = str(error)

split_colors = {"train": "#4477AA", "val": "#EEAA33", "test": "#CC3311"}
fig, axes = plt.subplots(
    1, 2, figsize=(18, 7), sharex=True, sharey=True, layout="constrained"
)
for ax, (method, _, _) in zip(axes, split_specs):
    result = split_results.get(method)
    if result is None:
        ax.scatter(molecules["x"], molecules["y"], color="0.8", s=30, alpha=0.5)
        ax.text(
            0.5, 0.5,
            "Split unavailable\n\n" + fill(split_errors[method], width=48),
            transform=ax.transAxes, ha="center", va="center", fontsize=10,
            bbox={"facecolor": "white", "alpha": 0.95, "edgecolor": "0.7"},
        )
    else:
        view = molecules[["id", "x", "y"]].merge(
            result[["id", "split"]], on="id", validate="one_to_one"
        )
        for split in SPLITS:
            subset = view[view["split"] == split]
            ax.scatter(
                subset["x"], subset["y"], color=split_colors[split],
                s=40, alpha=0.85, label=f"{split} ({len(subset)})",
            )
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    ax.set(title=f"{method} split assignments", xlabel=axis_labels[0], ylabel=axis_labels[1])
    if ANNOTATE_IDS:
        for row in molecules.itertuples():
            ax.annotate(
                str(row.id), (row.x, row.y), fontsize=6,
                xytext=(3, 3), textcoords="offset points",
            )
requested = "/".join(f"{fraction:.0%}" for fraction in SPLIT_FRAC)
fig.suptitle(f"Train / validation / test: {requested} | seed {SEED}\n{view_description}", fontsize=13)
plt.show()

if split_results:
    split_counts = pd.DataFrame({
        method: result["split"].value_counts().reindex(SPLITS, fill_value=0)
        for method, result in split_results.items()
    }).T
    split_counts.index.name = "method"
    display(split_counts)
for method, error in split_errors.items():
    print(f"{method}: {error}")